#**Step 4: Core 4-Step Training Loop**

#**Research/Learn:**

## Deep understanding of the four fundamental steps:
  * Forward pass: the process of passing input data through the network layers to produce predictions/outputs.
  
  * Loss calculation: quantifying how wrong the model's predictions are compared to the ground truth.
  
  * Backward pass:  computing gradients of the loss with respect to every parameter using the chain rule.
  
  * Optimization: using computed gradients to adjust weights in a direction that reduces loss
   
  ## How they aplly across model types:
  
  In training loop per batch, input datas go into Forward pass( prediction)-> loss calculation(scalar error value)-> backward pass (gradients for all params) -> optimize(updated weight) and repeat for next batch

## Use of accelerate for scaling:
Accelerate is a library by Hugging Face that makes distributed training simple by abstracting away hardware-specific code. With key Features:
  * Automatic Device Placement: handles all device placement
  * Mixed Precision Training: enable automatic mixed precision (FP16)
  * Accumulate gradients over multiple steps
  * Saving model


# Actions: Implement and compare high-level Trainer versus manual training loop.

Before implement and compare 2 approaches: I conduct to preprocess data. It mean I load, tokenize and remove useless column in dataset. The glue, mrpc dataset combine 4 columns: index, sentence1,sentence2, label. It suitable for classification model.


In [ ]:
from datasets import load_dataset
checkpoint = "bert-base-uncased"
raw_datasets= load_dataset("glue", "mrpc")

README.md: 0.00B [00:00, ?B/s]

mrpc/train-00000-of-00001.parquet:   0%|          | 0.00/649k [00:00<?, ?B/s]

mrpc/validation-00000-of-00001.parquet:   0%|          | 0.00/75.7k [00:00<?, ?B/s]

mrpc/test-00000-of-00001.parquet:   0%|          | 0.00/308k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

Check information of dataset

In [ ]:

raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]
raw_train_dataset.features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['not_equivalent', 'equivalent']),
 'idx': Value('int32')}

Define preprocessing tokenizer model

In [ ]:

from transformers import AutoTokenizer
tokenizer= AutoTokenizer.from_pretrained(checkpoint)


Preprocessing dataset: after processing, I have tokenizer_dataset include 7 columns.

In [ ]:

def tokenizer_function(example):
  return tokenizer(example["sentence1"],example["sentence2"],truncation=True)

tokenizer_dataset=raw_datasets.map(tokenizer_function,batched=True)



Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

Remove unnecessary columns. After preprocessing, the dataset contains four main columns:
- labels (target output)
- input_ids (vectorized representation of sentence1 and sentence2)
- attention_mask (indicates which tokens are real words and which are padding)
- token_type_ids (distinguishes tokens belonging to sentence1 and sentence2)

In [ ]:
tokenized_datasets = tokenizer_dataset.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

Loading model and data_collator

In [ ]:
from transformers import DataCollatorWithPadding
data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# =========================================================
# Metric
# =========================================================
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    # lấy class có xác suất cao nhất
    predictions = np.argmax(logits, axis=-1)

    # accuracy
    accuracy = accuracy_metric.compute(
        predictions=predictions,
        references=labels
    )

    # f1
    f1 = f1_metric.compute(
        predictions=predictions,
        references=labels
    )

    return {
        "accuracy": accuracy["accuracy"],
        "f1": f1["f1"]
    }

# Action 1: Implement high-level Trainer versus

In [ ]:
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np



# =========================================================
# TrainingArguments
# =========================================================
training_args = TrainingArguments(
    output_dir="./mrpc_bert_model",

    learning_rate=2e-5,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",
    # load_best_model_at_end=True,

    fp16=True,

    logging_steps=50,
    report_to="none"
)

# =========================================================
# Trainer
# =========================================================
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],

    data_collator=data_collator,
    processing_class=tokenizer,

    # thêm dòng này
    compute_metrics=compute_metrics
)

# Train
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.533806,0.448050,0.808824,0.873786
2,0.386210,0.417220,0.838235,0.890365
3,0.218307,0.424096,0.857843,0.896057
4,0.123118,0.570864,0.850490,0.894281
5,0.050323,0.634801,0.848039,0.893103


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1150, training_loss=0.260131892743318, metrics={'train_runtime': 242.0287, 'train_samples_per_second': 75.776, 'train_steps_per_second': 4.752, 'total_flos': 714950848507680.0, 'train_loss': 0.260131892743318, 'epoch': 5.0})

Result of training action 1: loss:0.04 after 5 epoches, train_runtime: 3mintues

#Action 2: Implement manual training loop for model

In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00


In [ ]:
 # initializate Accelerator
accelerator = Accelerator()


In [ ]:

# initialization  optimizer
optimizer = AdamW(model.parameters(), lr=2e-5)

In [ ]:
#  Prepare DataLoader
train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=16, collate_fn=data_collator
)
eval_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=16, collate_fn=data_collator
)

In [ ]:
#  Prepare accelerator
train_dl, eval_dl, model, optimizer = accelerator.prepare(
    train_dataloader, eval_dataloader, model, optimizer
)


In [ ]:
#  set up scheduler
num_epochs = 5
num_training_steps = num_epochs * len(train_dl)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

# set up metric
metric = evaluate.load("glue", "mrpc")

# set up progress bar
progress_bar = tqdm(range(num_training_steps))

  0%|          | 0/1150 [00:00<?, ?it/s]

In [ ]:
# training model:

model.train()
for epoch in range(num_epochs):
    total_train_loss = 0

    # Training phase
    for batch in train_dl:
        outputs = model(**batch)
        loss = outputs.loss

        # Backward pass (SỬ DỤNG ACCELERATOR)
        accelerator.backward(loss)

        total_train_loss += loss.item()

        # Update weights
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

        progress_bar.update(1)

    # Tính average loss
    avg_train_loss = total_train_loss / len(train_dl)

    print(f"\n--- Epoch {epoch + 1}/{num_epochs} Hoàn tất ---")
    print(f"Average Training Loss: {avg_train_loss:.4f}\n")

    #Evaluate
    model.eval()

    for batch in eval_dl:
        with torch.no_grad():
            outputs = model(**batch)

        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1)

        # Gather predictions từ tất cả devices (quan trọng cho multi-GPU)
        predictions, references = accelerator.gather_for_metrics(
            (predictions, batch["labels"])
        )

        metric.add_batch(predictions=predictions, references=references)

    # Compute metrics
    eval_metric = metric.compute()
    print(f"-> Validation Metrics: {eval_metric}")

    # Chuyển lại sang training mode cho epoch tiếp theo
    model.train()

print("\Done Training!")


<>:59: SyntaxWarning: invalid escape sequence '\D'
<>:59: SyntaxWarning: invalid escape sequence '\D'
/tmp/ipykernel_1723/540861013.py:59: SyntaxWarning: invalid escape sequence '\D'
  print("\Done Training!")



--- Epoch 1/5 Hoàn tất ---
Average Training Loss: 0.5559

-> Validation Metrics: {'accuracy': 0.8333333333333334, 'f1': 0.8831615120274914}

--- Epoch 2/5 Hoàn tất ---
Average Training Loss: 0.3382

-> Validation Metrics: {'accuracy': 0.8725490196078431, 'f1': 0.9081272084805654}

--- Epoch 3/5 Hoàn tất ---
Average Training Loss: 0.1725

-> Validation Metrics: {'accuracy': 0.8602941176470589, 'f1': 0.900523560209424}

--- Epoch 4/5 Hoàn tất ---
Average Training Loss: 0.0851

-> Validation Metrics: {'accuracy': 0.8602941176470589, 'f1': 0.9018932874354562}

--- Epoch 5/5 Hoàn tất ---
Average Training Loss: 0.0523

-> Validation Metrics: {'accuracy': 0.8578431372549019, 'f1': 0.9013605442176871}
\Done Training!


#Comparing 2 Approaches:
Two way have the same result. So that we consider about performance:
##Training Speed:
* Manual Trainng Loop: faster than Trainer API(2  minutes), but depends on implementation
* Trainer API(approximately 5 minutes): optimized automatically
## Flexibility:
* Manual Trainng Loop: very high, easy to customize
* Trainer API: moderate, limited customization
## Deployment Speed:
* Manual Trainng Loop:slower because more code is required
* Trainer API: faster and easier to deploy
## Support mixed precision:
* Manual Trainng Loop: Requires manual implementation or Accelerate
* Trainer API:Built-in support
## Suitable for:
* Manual Trainng Loop: research, experimentation, custom algorithms
* Trainer API: quick fine-tuning, benchmarking, production workflows

